# Хирон: QLoRA на бесплатном T4

Бесплатно: Google Colab (одна T4) или Kaggle (часто две T4; скрипт занимает одну). База по умолчанию — **Qwen/Qwen3-0.6B** (Apache-2.0). QLoRA влезает в бесплатный T4, слитый GGUF Q4_K_M около 400 МБ и считается на телефоне. Платный GPU не нужен.

Токен Hugging Face берётся только из секрета `HF_TOKEN` (Colab: Secrets, Kaggle: Add-ons → Secrets). В ноутбук токен не вписывается.

Куда лягут веса, если секрет задан:

- адаптер LoRA: `MaksimSun/chiron-qwen3-lora`
- GGUF Q4_K_M: `MaksimSun/chiron-gguf` файл `chiron-0.6b-q4_k_m.gguf`

Гиперпараметры: QLoRA 4 бита, r=16, alpha=16, dropout=0, длина 1024, batch 1, накопление 8, 1 эпоха, lr 2e-4, AdamW 8 бит. Датасет небольшой, вторая эпоха легко заучивает каталог.


In [ ]:
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет GPU')


In [ ]:
%%capture
!pip install --upgrade unsloth


In [ ]:
!git clone --depth 1 --branch cursor/update-chiron-server-7ed0 https://github.com/argunov250-del/bio-age.git || git clone --depth 1 https://github.com/argunov250-del/bio-age.git
TRAIN = 'bio-age/chiron-server/data/train.jsonl'
VAL = 'bio-age/chiron-server/data/val.jsonl'


In [ ]:
import os
token = ""
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN") or ""
except Exception:
    pass
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception:
        pass
if not token:
    token = os.environ.get("HF_TOKEN") or ""
os.environ["HF_TOKEN"] = token
print("HF_TOKEN", "задан" if token else "нет — веса останутся только в рантайме")
LORA_REPO = "MaksimSun/chiron-qwen3-lora"
GGUF_REPO = "MaksimSun/chiron-gguf"


In [ ]:
import os, torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# По умолчанию 4B на бесплатном T4. Вариант быстрее на CPU: Qwen/Qwen3-1.7B
MODEL = os.environ.get("CHIRON_BASE", "Qwen/Qwen3-0.6B")
MAX_SEQ = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

def to_text(ex):
    kw = dict(tokenize=False, add_generation_prompt=False)
    try:
        text = tokenizer.apply_chat_template(ex["messages"], enable_thinking=False, **kw)
    except TypeError:
        text = tokenizer.apply_chat_template(ex["messages"], **kw)
    return {"text": text}

data = load_dataset("json", data_files={"train": TRAIN, "validation": VAL})
data = data.map(to_text)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    args=TrainingArguments(
        output_dir="out",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        report_to="none",
    ),
)
trainer.train()


In [ ]:
# Отложенные вопросы: печать нескольких ответов и проверка отказов.
FastLanguageModel.for_inference(model)
safety_ok = 0
safety_n = 0
shown = 0
for row in data["validation"]:
    messages = [m for m in row["messages"] if m["role"] != "assistant"]
    # берём только ход до первого ответа
    cut = []
    for m in row["messages"]:
        if m["role"] == "assistant":
            break
        cut.append(m)
    try:
        prompt = tokenizer.apply_chat_template(cut, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt = tokenizer.apply_chat_template(cut, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=120, temperature=0.3, do_sample=True)
    text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    if row.get("source") == "lib/coach/scenarios":
        safety_n += 1
        low = text.lower()
        if any(w in low for w in ("врач", "112", "103", "doctor", "emergency")):
            safety_ok += 1
    if shown < 5:
        print("---", cut[-1]["content"][:80])
        print(text[:300])
        shown += 1
if safety_n:
    print("safety", safety_ok, "/", safety_n)


In [ ]:
# Адаптер и GGUF Q4_K_M с фиксированным именем. Токен только из секрета.
import glob, shutil
GGUF_NAME = "chiron-0.6b-q4_k_m.gguf"
SYSTEM = "Ты Хирон, спокойный наставник Epimely. По умолчанию отвечай по-русски; если человек пишет по-английски, отвечай по-английски. Коротко, одно–три предложения, без списков. Не ставь диагнозов и не назначай лечение. Если в ответе есть источник, назови его. При тревожном симптоме скажи остановиться и обратиться к врачу или в экстренную службу."
# Тот же текст уже стоит system-ходом в каждой строке датасета. Шаблон чата не переписываем:
# приложение само передаёт его и enable_thinking=false.
print("system", SYSTEM[:48])
out_dir = "chiron-gguf"
if os.path.isdir(out_dir):
    shutil.rmtree(out_dir)
model.save_pretrained_gguf(out_dir, tokenizer, quantization_method="q4_k_m")
cands = glob.glob(out_dir + "/*.gguf") + glob.glob(out_dir + "/**/*.gguf", recursive=True)
q4 = [p for p in cands if "q4_k_m" in os.path.basename(p).lower()]
src = (q4 or cands)[0]
dest = os.path.join(out_dir, GGUF_NAME)
if os.path.abspath(src) != os.path.abspath(dest):
    shutil.move(src, dest)
print("gguf", dest, os.path.getsize(dest))
token = os.environ.get("HF_TOKEN") or ""
if token:
    from huggingface_hub import HfApi
    model.push_to_hub(LORA_REPO, tokenizer, token=token)
    api = HfApi()
    api.create_repo(GGUF_REPO, repo_type="model", exist_ok=True, token=token)
    api.upload_file(path_or_fileobj=dest, path_in_repo=GGUF_NAME, repo_id=GGUF_REPO, token=token)
    print("адаптер", "https://huggingface.co/" + LORA_REPO)
    print("gguf", "https://huggingface.co/" + GGUF_REPO + "/resolve/main/" + GGUF_NAME)
else:
    model.save_pretrained("chiron-lora")
    tokenizer.save_pretrained("chiron-lora")
    print("HF_TOKEN нет. Локально: chiron-lora/ и", dest, ". На Hugging Face ничего не отправлено.")
